# **🛩️ ATAS**

The ATAS (Airial Threat Assesment System) is an end-to-end machine learning engineering project that combines

- computer vision,
- structured machine learning,
- tactical scenario generation,
- and interactive visualization into a single pipeline.

**The project is divided into three major components:**

- Aircraft Classification Model
- ETA Regressor Model
- Hit Classification Model

---

**In this notebook**

# **🎯 Hit Probability Classifier**

This notebook focuses on building the Hit Probability Classifier using `scikit-learn` and `ensemble methods` to predict whether a missile will hit your aircraft after an evasion attempt, across `1,000,000` synthetic engagement scenarios.

## **1. Problem Definition**

Physics can tell you where a missile is going, that is simple trajectory calculation.

What it cannot tell you is **whether the missile will still hit you after you maneuver.**

Given the engagement conditions including aircraft states, missile specifications, 
geometry, and countermeasure state, predict **whether the missile hits your aircraft 
after an evasion attempt** → a binary outcome of **hit or miss**.

This outcome depends on non-linear interactions between aspect angle, maneuverability, 
missile phase, and countermeasure effectiveness, making it a genuine ML problem, 
not something physics equations can resolve.

---

## **2. Data**

The data used here is synthetic, generated using real physics equations since 
real labeled engagement data does not exist publicly. This is standard practice; 
companies like Tesla and Waymo also train models on synthetic data.

**How the data was made:**
- Generated by `physics_generator.py` using real aircraft specs from `aircraft_metadata.csv`
- Stored as `synthetic_engagements.csv`

**Key notes on Data**
- The data contains 16 columns, 1,000,000 rows
- **14 columns are input features:**
  
| Feature | Description |
|---|---|
| `launch_distance` | Distance from which the missile was fired (metres) |
| `remaining_distance` | Current distance of missile from your aircraft |
| `closure_rate` | Combined closing speed of missile + your aircraft (m/s) |
| `azimuth` | Horizontal angle of incoming threat (0°–360°) |
| `elevation` | Vertical angle of incoming threat (-90°–+90°) |
| `missile_phase` | 0 = boost, 1 = mid-course, 2 = terminal |
| `your_speed` | Your current airspeed (m/s) |
| `your_altitude` | Your current altitude (metres) |
| `your_maneuverability` | 0 = low, 1 = medium, 2 = high |
| `enemy_altitude` | Enemy aircraft altitude (metres) |
| `missile_speed` | Speed of incoming missile (m/s) |
| `missile_range` | Max effective range of missile (metres) |
| `enemy_generation` | Aircraft generation (3.5, 4, 4.5, 5) |
| `countermeasure_deployed` | 0 = not deployed, 1 = deployed |
  
- **2 columns are target labels:**
  
| Label | Description |
|---|---|
| `evasion_time` | Minimum time available to evade the missile (seconds) - dropped here |
| `hit` | Whether the missile hits after evasion (0 = miss, 1 = hit) - **target for this model** |

---

## **3. Evaluation**

The primary evaluation metric for this model is **Recall** - it tells us how 
often the model correctly identifies missiles that will actually hit, which is 
directly meaningful in a tactical context.

Missing a real hit (false negative) is far more dangerous than a false alarm 
(false positive). The model should never tell a pilot they are safe when they 
are not.

**All three metrics will be tracked:**

| Metric | Goal |
|---|---|
| Recall | > 0.90 — minimise missed hits |
| F1 Score | > 0.85 — balance between precision and recall |
| ROC-AUC | > 0.95 — overall discrimination ability |

---

> Note: Recall is prioritised over precision here. In a tactical system, a 
false alarm costs a countermeasure. A missed hit costs a life.

> These targets are preliminary. Final metric selection and thresholds will be 
confirmed after EDA once the class balance of the `hit` column is known. 
If the dataset is heavily imbalanced, additional metrics may be introduced.

---

## **4. Features**

Some information about the data:

* We are dealing with structured tabular data so we will be using ensemble methods (Random Forest, XGBoost).
* The dataset contains `1,000,000` synthetic engagement scenarios across `101` aircraft types.
* All 14 input features are numerical - no text, no images, no missing values.
* Features span four groups:
   * **Engagement geometry:** `launch_distance`, `remaining_distance`, `closure_rate`, `azimuth`, `elevation`, `missile_phase`
   * **Your aircraft state:** `your_speed`, `your_altitude`, `your_maneuverability`
   * **Enemy aircraft state:** `enemy_altitude`
   * **Threat specs:** `missile_speed`, `missile_range`, `enemy_generation`, `countermeasure_deployed`
* The target variable is `hit` (binary, 0 = miss, 1 = hit) → the `evasion_time` column is dropped.
* Data was generated from physics rules — class balance of the `hit` column will be confirmed during EDA.
  
* Features like `aspect_angle`, `your_maneuverability`, and `countermeasure_deployed` introduce non-linear interactions that make this a genuine ML problem rather than a simple trajectory calculation.

---

## **5. Preparing the Tools**

We will be using the following libraries for this project:

* **NumPy** → numerical operations
* **Pandas** → loading and manipulating the synthetic engagement dataset
* **Matplotlib** → visualizing feature distributions and model performance
* **Scikit-learn** → model training, evaluation metrics, and train/test split
* **XGBoost** → gradient boosted trees (primary candidate)
* **Joblib** → saving and loading trained models

In [1]:
# Importing the Librabries
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import xgboost
import joblib

%matplotlib inline

In [2]:
# Use for naming the files
import datetime, pytz
ist = pytz.timezone("Asia/Kolkata")
datetime.datetime.now(ist).strftime('%Y-%m-%d_%H-%M-%S')

'2026-06-02_17-25-51'

In [3]:
import os
from pathlib import Path

# Set to:
# - "auto"   -> detect the runtime
# - "colab"  -> force Google Colab
# - "kaggle" -> force Kaggle
# - "local"  -> force local machine
# - "lightning" -> force Lightning AI (workspace)

FORCE_ENV = "auto"

def detect_env():
    if os.path.exists("/teamspace/studios"):
        return "lightning"
    if os.path.exists("/kaggle"):
        return "kaggle"
    if "COLAB_GPU" in os.environ or os.path.exists("/content/drive"):
        return "colab"
    return "local"

def find_local_project_root(start: Path) -> Path:
    """Find project root by walking up to a folder that contains environment.yml."""
    for candidate in [start, *start.parents]:
        if (candidate / "environment.yml").exists():
            return candidate
    return start

ENV = detect_env() if FORCE_ENV == "auto" else FORCE_ENV

if ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/ATAS_Project")
    DATA_ROOT = PROJECT_ROOT / "data"
elif ENV == "kaggle":
    PROJECT_ROOT = Path("/kaggle/working")
    DATA_ROOT = Path("/kaggle/input/datasets/eakempreetsingh/atas-aircraft-dataset")
    if not DATA_ROOT.exists():
        DATA_ROOT = PROJECT_ROOT / "data"
elif ENV == "lightning":
    PROJECT_ROOT = Path("/teamspace/studios/this_studio")
    DATA_ROOT = PROJECT_ROOT / "data"
else:
    PROJECT_ROOT = find_local_project_root(Path.cwd().resolve())
    DATA_ROOT = PROJECT_ROOT / "data"

# Standard paths
LOG_DIR = PROJECT_ROOT / "logs"
MODEL_DIR = PROJECT_ROOT / "models"
RELEASE_MODELS_DIR = PROJECT_ROOT / "release_models"
PREDICTIONS_DIR = PROJECT_ROOT / "predictions"

# ETA-specific path
HIT_DATA_ROOT = DATA_ROOT / "hit"

# Create folders safely (never inside read-only Kaggle input)
for folder_path in [PROJECT_ROOT, DATA_ROOT, LOG_DIR, MODEL_DIR, RELEASE_MODELS_DIR, PREDICTIONS_DIR, HIT_DATA_ROOT]:
    if ENV == "kaggle" and str(folder_path).startswith("/kaggle/input"):
        continue
    try:
        os.makedirs(folder_path, exist_ok=True)
    except PermissionError:
        pass

print(f"ENV:                {ENV}")
print(f"PROJECT_ROOT:       {PROJECT_ROOT}")
print(f"DATA_ROOT:          {DATA_ROOT}")
print(f"HIT_DATA_ROOT:      {HIT_DATA_ROOT}")
print(f"RELEASE_MODELS_DIR: {RELEASE_MODELS_DIR}")

ENV:                local
PROJECT_ROOT:       /mnt/d/Eakem/Learning ML and DS/Project_ATAS/ATAS_Project
DATA_ROOT:          /mnt/d/Eakem/Learning ML and DS/Project_ATAS/ATAS_Project/data
HIT_DATA_ROOT:      /mnt/d/Eakem/Learning ML and DS/Project_ATAS/ATAS_Project/data/hit
RELEASE_MODELS_DIR: /mnt/d/Eakem/Learning ML and DS/Project_ATAS/ATAS_Project/release_models


## **6. Importing the data and preparing it for modelling**

In [4]:
# Confirm resolved project paths (do not overwrite them here)
PROJECT_ROOT, DATA_ROOT, HIT_DATA_ROOT

(PosixPath('/mnt/d/Eakem/Learning ML and DS/Project_ATAS/ATAS_Project'),
 PosixPath('/mnt/d/Eakem/Learning ML and DS/Project_ATAS/ATAS_Project/data'),
 PosixPath('/mnt/d/Eakem/Learning ML and DS/Project_ATAS/ATAS_Project/data/hit'))

In [12]:
# Loading the dataframe from environment-aware data root
aircraft_df = pd.read_csv(DATA_ROOT / "synthetic_engagements.csv",
                          low_memory=False)

# # Drop the evasion_time column for Hit Classifier
# aircraft_df = aircraft_df.drop(columns="evasion_time")
# aircraft_df.head()

# Will drop after dropping rows who has eta more than 300sec 

In [13]:
# Drop physically unrealistic scenarios (same filter applied in ETA regressor)
aircraft_df = aircraft_df[aircraft_df["evasion_time"] <= 300].reset_index(drop=True)

# Now drop evasion_time - not needed for hit classifier
aircraft_df = aircraft_df.drop(columns="evasion_time")

print(aircraft_df.shape)
aircraft_df.head()

(999405, 15)


,launch_distance,remaining_distance,closure_rate,azimuth,elevation,missile_phase,your_speed,your_altitude,your_maneuverability,enemy_altitude,missile_speed,missile_range,enemy_generation,countermeasure_deployed,hit
0,33895.443880,5709.789560,795.851952,110.969042,26.933223,2,191.657942,20767.395530,2,15722.748320,857,35000,4.0,1,0
1,9775.992373,1963.280302,846.016340,91.346829,-24.938668,2,515.353333,24375.588350,2,10976.820090,857,35000,4.0,0,0
2,24382.997910,7464.370311,829.119185,127.394632,47.364627,2,67.779862,21557.988950,0,6022.335489,857,35000,4.0,1,0
3,28058.181280,17335.851080,868.412502,305.274787,81.459572,1,133.070710,7500.712655,2,18492.264380,857,35000,4.0,1,0
4,22448.079020,9829.917985,517.446433,199.804139,32.414351,1,427.506675,4408.181588,1,11524.183400,857,35000,4.0,1,0


In [14]:
# length of dataset
len(aircraft_df)

999405

In [15]:
# Shape of dataset
aircraft_df.shape

(999405, 15)

In [16]:
# Feature value distribution
aircraft_df.describe()

,launch_distance,remaining_distance,closure_rate,azimuth,elevation,missile_phase,your_speed,your_altitude,your_maneuverability,enemy_altitude,missile_speed,missile_range,enemy_generation,countermeasure_deployed,hit
count,999405.000000,999405.000000,999405.000000,999405.000000,999405.000000,999405.000000,999405.000000,999405.000000,999405.00000,999405.000000,999405.000000,999405.000000,999405.000000,999405.000000,999405.000000
mean,65013.755552,32542.830822,1276.555065,179.937820,0.025930,1.009734,520.993280,14984.533823,1.00115,15009.353638,1274.093497,129474.970608,4.284411,0.500388,0.407376
std,74170.064429,46805.403792,467.511980,104.078646,52.051722,0.818450,265.924147,8666.247323,0.81667,8657.791088,370.223841,111043.200468,0.456509,0.500000,0.491346
min,500.024387,0.002412,0.206626,0.001255,-89.999940,0.000000,61.000864,0.008947,0.00000,0.036823,686.000000,8000.000000,3.500000,0.000000,0.000000
25%,9739.140529,3404.931449,907.111492,89.627150,-45.155190,0.000000,290.638551,7468.446199,0.00000,7532.486649,857.000000,35000.000000,4.000000,0.000000,0.000000
50%,37354.954850,13994.035580,1324.890219,179.910579,0.032270,1.000000,520.832276,14971.837290,1.00000,14998.463830,1372.000000,110000.000000,4.000000,1.000000,0.000000
75%,94130.643570,42330.875730,1569.283500,270.177444,45.171235,2.000000,750.908502,22489.231270,2.00000,22511.722400,1372.000000,160000.000000,4.500000,1.000000,1.000000
max,399998.390200,397823.101500,3033.120274,359.998621,89.999688,2.000000,982.998755,29999.961160,2.00000,29999.936050,2058.000000,400000.000000,5.000000,1.000000,1.000000


In [17]:
aircraft_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 999405 entries, 0 to 999404
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   launch_distance          999405 non-null  float64
 1   remaining_distance       999405 non-null  float64
 2   closure_rate             999405 non-null  float64
 3   azimuth                  999405 non-null  float64
 4   elevation                999405 non-null  float64
 5   missile_phase            999405 non-null  int64  
 6   your_speed               999405 non-null  float64
 7   your_altitude            999405 non-null  float64
 8   your_maneuverability     999405 non-null  int64  
 9   enemy_altitude           999405 non-null  float64
 10  missile_speed            999405 non-null  int64  
 11  missile_range            999405 non-null  int64  
 12  enemy_generation         999405 non-null  float64
 13  countermeasure_deployed  999405 non-null  int64  
 14  hit            

In [18]:
# Checking for any null values
aircraft_df.isna().sum()

launch_distance            0
remaining_distance         0
closure_rate               0
azimuth                    0
elevation                  0
missile_phase              0
your_speed                 0
your_altitude              0
your_maneuverability       0
enemy_altitude             0
missile_speed              0
missile_range              0
enemy_generation           0
countermeasure_deployed    0
hit                        0
dtype: int64

In [19]:
# Looking at hit(1) and miss(0) values
aircraft_df["hit"].value_counts()


hit
0    592271
1    407134
Name: count, dtype: int64

**Data is loaded, filtered to 999,405 rows by removing physically unrealistic 
scenarios above 300 seconds, matching the same distribution the ETA regressor 
was trained on.**

**The `evasion_time` column is dropped since this model only 
predicts `hit`.**

**14 features remain, no nulls, no missing values, ready for exploration.**

---